### ETL + BI

In [2]:
#Install libraries
import requests
import pandas as pd

## Task 01: Data ingestion from an API - Extraction

In [3]:
#Get data from the endpoints
# Pull Products
data = requests.get("https://northwind.vercel.app/api/products").json()
products = pd.DataFrame(data)

# Pull Orders
data2 = requests.get("https://northwind.vercel.app/api/orders").json()
orders = pd.DataFrame(data2)

# Pull Customers
data3 = requests.get("https://northwind.vercel.app/api/customers").json()
customers = pd.DataFrame(data3)

#pull employess
data4 = requests.get("https://northwind.vercel.app/api/employees").json()
employees = pd.DataFrame(data4)

#Pull suppliers
data5 = requests.get("https://northwind.vercel.app/api/suppliers").json()
suppliers = pd.DataFrame(data5)

#pull order details
data6 = requests.get("https://northwind.vercel.app/api/order-details").json()
Order_details = pd.DataFrame(data6)

#pull shippers
data7 = requests.get("https://northwind.vercel.app/api/shippers").json()
shippers = pd.DataFrame(data7)


## Task 02 : Data Transformation using Pandas and Numpy

In [4]:
products.head()

,id,supplierId,categoryId,quantityPerUnit,unitPrice,unitsInStock,unitsOnOrder,reorderLevel,discontinued,name,supplier,category
0,4,2,3,48 - 6 oz jars,22.00,53,0,0,True,Chef Anton's Cajun Seasoning,"{'id': 2, 'companyName': 'New Orleans Cajun De...","{'id': 3, 'description': 'Desserts candies and..."
1,5,2,2,36 boxes,21.35,0,0,0,True,Chef Anton's Gumbo Mix,NaN,NaN
2,6,3,2,12 - 8 oz jars,25.00,120,0,25,False,Grandma's Boysenberry Spread,NaN,NaN
3,7,3,7,12 - 1 lb pkgs.,30.00,15,0,10,False,Uncle Bob's Organic Dried Pears,NaN,NaN
4,8,3,2,12 - 12 oz jars,40.00,6,0,0,False,Northwoods Cranberry Sauce,NaN,NaN


In [5]:
orders.head()

,id,customerId,employeeId,orderDate,requiredDate,shippedDate,shipVia,freight,shipName,shipAddress,details
0,10271,SPLIR,6,1996-08-01 00:00:00.000,1996-08-29 00:00:00.000,1996-08-30 00:00:00.000,2,4.54,Split Rail Beer & Ale,"{'street': 'P.O. Box 555', 'city': 'Lander', '...","[{'productId': 33, 'unitPrice': 2, 'quantity':..."
1,10266,WARTH,3,1996-07-26 00:00:00.000,1996-09-06 00:00:00.000,1996-07-31 00:00:00.000,3,25.73,Wartian Herkku,"{'street': 'Torikatu 38', 'city': 'Oulu', 'reg...","[{'productId': 12, 'unitPrice': 30.4, 'quantit..."
2,10279,LEHMS,8,1996-08-13 00:00:00.000,1996-09-10 00:00:00.000,1996-08-16 00:00:00.000,2,25.83,Lehmanns Marktstand,"{'street': 'Magazinweg 7', 'city': 'Frankfurt ...","[{'productId': 17, 'unitPrice': 31.2, 'quantit..."
3,10292,TRADH,1,1996-08-28 00:00:00.000,1996-09-25 00:00:00.000,1996-09-02 00:00:00.000,2,1.35,Tradiçao Hipermercados,"{'street': 'Av. Inês de Castro 414', 'city': '...","[{'productId': 20, 'unitPrice': 64.8, 'quantit..."
4,10295,VINET,2,1996-09-02 00:00:00.000,1996-09-30 00:00:00.000,1996-09-10 00:00:00.000,2,1.15,Vins et alcools Chevalier,"{'street': '59 rue de l'Abbaye', 'city': 'Reim...","[{'productId': 56, 'unitPrice': 30.4, 'quantit..."


## Orders

In [6]:
# Example: expand shipAddress to columns
ship_address = pd.json_normalize(orders['shipAddress'])
ship_address.columns = [f"ship_{col}" for col in ship_address.columns]

# Combine back
df_orders= pd.concat([orders.drop('shipAddress', axis=1), ship_address], axis=1)

df_orders.head()

,id,customerId,employeeId,orderDate,requiredDate,shippedDate,shipVia,freight,shipName,details,ship_street,ship_city,ship_region,ship_postalCode,ship_country
0,10271,SPLIR,6,1996-08-01 00:00:00.000,1996-08-29 00:00:00.000,1996-08-30 00:00:00.000,2,4.54,Split Rail Beer & Ale,"[{'productId': 33, 'unitPrice': 2, 'quantity':...",P.O. Box 555,Lander,WY,82520,USA
1,10266,WARTH,3,1996-07-26 00:00:00.000,1996-09-06 00:00:00.000,1996-07-31 00:00:00.000,3,25.73,Wartian Herkku,"[{'productId': 12, 'unitPrice': 30.4, 'quantit...",Torikatu 38,Oulu,NULL,90110,Finland
2,10279,LEHMS,8,1996-08-13 00:00:00.000,1996-09-10 00:00:00.000,1996-08-16 00:00:00.000,2,25.83,Lehmanns Marktstand,"[{'productId': 17, 'unitPrice': 31.2, 'quantit...",Magazinweg 7,Frankfurt a.M.,NULL,60528,Germany
3,10292,TRADH,1,1996-08-28 00:00:00.000,1996-09-25 00:00:00.000,1996-09-02 00:00:00.000,2,1.35,Tradiçao Hipermercados,"[{'productId': 20, 'unitPrice': 64.8, 'quantit...",Av. Inês de Castro 414,Sao Paulo,SP,05634-030,Brazil
4,10295,VINET,2,1996-09-02 00:00:00.000,1996-09-30 00:00:00.000,1996-09-10 00:00:00.000,2,1.15,Vins et alcools Chevalier,"[{'productId': 56, 'unitPrice': 30.4, 'quantit...",59 rue de l'Abbaye,Reims,NULL,51100,France


In [7]:
#Drop the details column
df_orders = df_orders.drop('details', axis=1)

#Rename columns
df_orders = df_orders.rename(columns={
    "id": "order_id",
    "customerId": "customer_id",
    "employeeId": "employee_id",
    "orderDate": "order_date",
    "requiredDate": "required_date",
    "shippedDate": "shipped_date",
    "shipVia": "ship_via",
    "freight": "freight_cost",
    "shipName": "ship_name"
})

df_orders.head()

,order_id,customer_id,employee_id,order_date,required_date,shipped_date,ship_via,freight_cost,ship_name,ship_street,ship_city,ship_region,ship_postalCode,ship_country
0,10271,SPLIR,6,1996-08-01 00:00:00.000,1996-08-29 00:00:00.000,1996-08-30 00:00:00.000,2,4.54,Split Rail Beer & Ale,P.O. Box 555,Lander,WY,82520,USA
1,10266,WARTH,3,1996-07-26 00:00:00.000,1996-09-06 00:00:00.000,1996-07-31 00:00:00.000,3,25.73,Wartian Herkku,Torikatu 38,Oulu,NULL,90110,Finland
2,10279,LEHMS,8,1996-08-13 00:00:00.000,1996-09-10 00:00:00.000,1996-08-16 00:00:00.000,2,25.83,Lehmanns Marktstand,Magazinweg 7,Frankfurt a.M.,NULL,60528,Germany
3,10292,TRADH,1,1996-08-28 00:00:00.000,1996-09-25 00:00:00.000,1996-09-02 00:00:00.000,2,1.35,Tradiçao Hipermercados,Av. Inês de Castro 414,Sao Paulo,SP,05634-030,Brazil
4,10295,VINET,2,1996-09-02 00:00:00.000,1996-09-30 00:00:00.000,1996-09-10 00:00:00.000,2,1.15,Vins et alcools Chevalier,59 rue de l'Abbaye,Reims,NULL,51100,France


In [8]:
import numpy as np

# Replace 'NULL' strings with actual NaN
df_orders = df_orders.replace('NULL', np.nan)

In [9]:
#Fill null values
df_orders["ship_region"] = df_orders["ship_region"].fillna("N/A")
df_orders["ship_postalCode"] = df_orders["ship_postalCode"].fillna("N/A")


#Check for null data
print(df_orders.isnull().sum())

order_id            0
customer_id         0
employee_id         0
order_date          0
required_date       0
shipped_date       21
ship_via            0
freight_cost        0
ship_name           0
ship_street         0
ship_city           0
ship_region         0
ship_postalCode     0
ship_country        0
dtype: int64


In [10]:
#parse dates to datetime
df_orders["order_date"] = pd.to_datetime(df_orders["order_date"])
df_orders["required_date"] = pd.to_datetime(df_orders["required_date"])
df_orders["shipped_date"] = pd.to_datetime(df_orders["shipped_date"])

df_orders.head()

,order_id,customer_id,employee_id,order_date,required_date,shipped_date,ship_via,freight_cost,ship_name,ship_street,ship_city,ship_region,ship_postalCode,ship_country
0,10271,SPLIR,6,1996-08-01,1996-08-29,1996-08-30,2,4.54,Split Rail Beer & Ale,P.O. Box 555,Lander,WY,82520,USA
1,10266,WARTH,3,1996-07-26,1996-09-06,1996-07-31,3,25.73,Wartian Herkku,Torikatu 38,Oulu,N/A,90110,Finland
2,10279,LEHMS,8,1996-08-13,1996-09-10,1996-08-16,2,25.83,Lehmanns Marktstand,Magazinweg 7,Frankfurt a.M.,N/A,60528,Germany
3,10292,TRADH,1,1996-08-28,1996-09-25,1996-09-02,2,1.35,Tradiçao Hipermercados,Av. Inês de Castro 414,Sao Paulo,SP,05634-030,Brazil
4,10295,VINET,2,1996-09-02,1996-09-30,1996-09-10,2,1.15,Vins et alcools Chevalier,59 rue de l'Abbaye,Reims,N/A,51100,France


## Customers

In [11]:
customers.head()

,id,companyName,contactName,contactTitle,address
0,AROUT,Around the Horn,Thomas Hardy,Sales Representative,"{'street': '120 Hanover Sq.', 'city': 'London'..."
1,BERGS,Berglunds snabbköp,Christina Berglund,Order Administrator,"{'street': 'Berguvsvägen 8', 'city': 'Luleå',..."
2,BLAUS,Blauer See Delikatessen,Hanna Moos,Sales Representative,"{'street': 'Forsterstr. 57', 'city': 'Mannheim..."
3,BLONP,Blondesddsl père et fils,Frédérique Citeaux,Marketing Manager,"{'street': '24 place Kléber', 'city': 'Strasbo..."
4,BOLId,Bólido Comidas preparadas,Martín Sommer,Owner,"{'street': '67C Araquil', 'city': 'Madrid', 'r..."


In [12]:
address = pd.json_normalize(customers["address"])
address.columns = [f"address_{col}" for col in address.columns]

customers= pd.concat([customers.drop("address", axis=1), address], axis=1)

customers.head()


,id,companyName,contactName,contactTitle,address_street,address_city,address_region,address_postalCode,address_country,address_phone
0,AROUT,Around the Horn,Thomas Hardy,Sales Representative,120 Hanover Sq.,London,NULL,WA1 1DP,UK,(171) 555-7788
1,BERGS,Berglunds snabbköp,Christina Berglund,Order Administrator,Berguvsvägen 8,Luleå,NULL,S-958 22,Sweden,0921-12 34 65
2,BLAUS,Blauer See Delikatessen,Hanna Moos,Sales Representative,Forsterstr. 57,Mannheim,NULL,68306,Germany,0621-08460
3,BLONP,Blondesddsl père et fils,Frédérique Citeaux,Marketing Manager,24 place Kléber,Strasbourg,NULL,67000,France,88.60.15.31
4,BOLId,Bólido Comidas preparadas,Martín Sommer,Owner,67C Araquil,Madrid,NULL,28023,Spain,(91) 555 22 82


In [13]:
# Replace 'NULL' strings with actual NaN
customers = customers.replace('NULL', np.nan)

print(customers.isnull().sum())

id                     0
companyName            0
contactName            0
contactTitle           0
address_street         0
address_city           0
address_region        60
address_postalCode     1
address_country        0
address_phone          0
dtype: int64


In [15]:
# Fill NaN with N/A
customers["address_region"] = customers["address_region"].fillna("N/A")
customers["address_postalCode"] = customers["address_postalCode"].fillna("N/A")

# #Rename columns
# df_customers = customer.rename(columns={
#     "id": "customer_i",
#     "companyName": "company_name",
#     "contactName": "contact_name",
#     "contactTitle": "contact_title",
#     "address_street": "street",
#     "address_city": "city",
#     "region": "region",
#     "postal_code": "postal_code",
#     "address_country": "country",
#     "address_phone": "phone"
# })


customers.head()

,id,companyName,contactName,contactTitle,address_street,address_city,address_region,address_postalCode,address_country,address_phone
0,AROUT,Around the Horn,Thomas Hardy,Sales Representative,120 Hanover Sq.,London,N/A,WA1 1DP,UK,(171) 555-7788
1,BERGS,Berglunds snabbköp,Christina Berglund,Order Administrator,Berguvsvägen 8,Luleå,N/A,S-958 22,Sweden,0921-12 34 65
2,BLAUS,Blauer See Delikatessen,Hanna Moos,Sales Representative,Forsterstr. 57,Mannheim,N/A,68306,Germany,0621-08460
3,BLONP,Blondesddsl père et fils,Frédérique Citeaux,Marketing Manager,24 place Kléber,Strasbourg,N/A,67000,France,88.60.15.31
4,BOLId,Bólido Comidas preparadas,Martín Sommer,Owner,67C Araquil,Madrid,N/A,28023,Spain,(91) 555 22 82


In [16]:
# #Rename columns
df_customers = customers.rename(columns={
    "id": "customer_id",
    "companyName": "company_name",
    "contactName": "contact_name",
    "contactTitle": "contact_title",
    "address_street": "street",
    "address_city": "city",
    "region": "region",
    "postal_code": "postal_code",
    "address_country": "country",
    "address_phone": "phone"
 })


df_customers.head()

,customer_id,company_name,contact_name,contact_title,street,city,address_region,address_postalCode,country,phone
0,AROUT,Around the Horn,Thomas Hardy,Sales Representative,120 Hanover Sq.,London,N/A,WA1 1DP,UK,(171) 555-7788
1,BERGS,Berglunds snabbköp,Christina Berglund,Order Administrator,Berguvsvägen 8,Luleå,N/A,S-958 22,Sweden,0921-12 34 65
2,BLAUS,Blauer See Delikatessen,Hanna Moos,Sales Representative,Forsterstr. 57,Mannheim,N/A,68306,Germany,0621-08460
3,BLONP,Blondesddsl père et fils,Frédérique Citeaux,Marketing Manager,24 place Kléber,Strasbourg,N/A,67000,France,88.60.15.31
4,BOLId,Bólido Comidas preparadas,Martín Sommer,Owner,67C Araquil,Madrid,N/A,28023,Spain,(91) 555 22 82


## Shippers

In [17]:
shippers.head()

,id,companyName,phone
0,1,Speedy Express,(503) 555-9831
1,2,United Package,(503) 555-3199
2,3,Federal Shipping,(503) 555-9931


In [ ]:
#Check for null values
print(shippers.isnull().sum())

id             0
companyName    0
phone          0
dtype: int64


In [ ]:
#Rename columns
df_shippers= shippers.rename(columns={
    "id": "shipper_id",
    "companyName": "company_name",
    "phone": "phone"
})

df_shippers.tail()


,shipper_id,company_name,phone
0,1,Speedy Express,(503) 555-9831
1,2,United Package,(503) 555-3199
2,3,Federal Shipping,(503) 555-9931


## Suppliers

In [18]:
suppliers.head()

,id,companyName,contactName,contactTitle,address
0,4,Tokyo Traders,Yoshi Nagase,Marketing Manager,"{'street': '9-8 Sekimai Musashino-shi', 'city'..."
1,5,Cooperativa de Quesos 'Las Cabras',Antonio del Valle Saavedra,Export Administrator,"{'street': 'Calle del Rosal 4', 'city': 'Ovied..."
2,6,Mayumi's,Mayumi Ohno,Marketing Representative,"{'street': '92 Setsuko Chuo-ku', 'city': 'Osak..."
3,7,Pavlova Ltd.,Ian Devling,Marketing Manager,"{'street': '74 Rose St. Moonie Ponds', 'city':..."
4,8,Specialty Biscuits Ltd.,Peter Wilson,Sales Representative,"{'street': '29 King's Way', 'city': 'Mancheste..."


In [24]:
#Expand the nested address column
address = pd.json_normalize(suppliers["address"])
address.columns = [f"address_{col}" for col in address.columns]

suppliers = pd.concat([suppliers.drop("address", axis=1), address], axis=1)

suppliers.head()


,id,companyName,contactName,contactTitle,address_street,address_city,address_region,address_postalCode,address_country,address_phone
0,4,Tokyo Traders,Yoshi Nagase,Marketing Manager,9-8 Sekimai Musashino-shi,Tokyo,NULL,100,Japan,(03) 3555-5011
1,5,Cooperativa de Quesos 'Las Cabras',Antonio del Valle Saavedra,Export Administrator,Calle del Rosal 4,Oviedo,Asturias,33007,Spain,(98) 598 76 54
2,6,Mayumi's,Mayumi Ohno,Marketing Representative,92 Setsuko Chuo-ku,Osaka,NULL,545,Japan,(06) 431-7877
3,7,Pavlova Ltd.,Ian Devling,Marketing Manager,74 Rose St. Moonie Ponds,Melbourne,Victoria,3058,Australia,(03) 444-2343
4,8,Specialty Biscuits Ltd.,Peter Wilson,Sales Representative,29 King's Way,Manchester,NULL,M14 GSD,UK,(161) 555-4448


In [27]:
# Replace 'NULL' strings with actual NaN
suppliers = suppliers.replace('NULL', np.nan)
suppliers.head()

,id,companyName,contactName,contactTitle,address_street,address_city,address_region,address_postalCode,address_country,address_phone
0,4,Tokyo Traders,Yoshi Nagase,Marketing Manager,9-8 Sekimai Musashino-shi,Tokyo,NaN,100,Japan,(03) 3555-5011
1,5,Cooperativa de Quesos 'Las Cabras',Antonio del Valle Saavedra,Export Administrator,Calle del Rosal 4,Oviedo,Asturias,33007,Spain,(98) 598 76 54
2,6,Mayumi's,Mayumi Ohno,Marketing Representative,92 Setsuko Chuo-ku,Osaka,NaN,545,Japan,(06) 431-7877
3,7,Pavlova Ltd.,Ian Devling,Marketing Manager,74 Rose St. Moonie Ponds,Melbourne,Victoria,3058,Australia,(03) 444-2343
4,8,Specialty Biscuits Ltd.,Peter Wilson,Sales Representative,29 King's Way,Manchester,NaN,M14 GSD,UK,(161) 555-4448


In [33]:
print(suppliers.isnull().sum())

id                    0
companyName           0
contactName           0
contactTitle          0
address_street        0
address_city          0
address_region        0
address_postalCode    0
address_country       0
address_phone         0
dtype: int64


In [32]:
#Fill null values with N/A
suppliers["contactTitle"] = suppliers["contactTitle"].fillna("N/A")
suppliers["address_street"] = suppliers["address_street"].fillna("N/A")
suppliers["address_city"] = suppliers["address_city"].fillna("N/A")
suppliers["address_region"] = suppliers["address_region"].fillna("N/A")
suppliers["address_postalCode"] = suppliers["address_postalCode"].fillna("N/A")
suppliers["address_country"] = suppliers["address_country"].fillna("N/A")



In [34]:
#Rename columns
df_suppliers = suppliers.rename(columns={
    "id": "supplier_id",
    "companyName": "company_name",
    "contactName": "contact_name",
    "contactTitle": "contact_title",
    "address_street": "street",
    "address_city": "city",
    "address_region": "region",
    "address_postalCode": "postal_code",
    "address_country": "country",
    "address_phone": "phone"
})
df_suppliers.head()

,supplier_id,company_name,contact_name,contact_title,street,city,region,postal_code,country,phone
0,4,Tokyo Traders,Yoshi Nagase,Marketing Manager,9-8 Sekimai Musashino-shi,Tokyo,N/A,100,Japan,(03) 3555-5011
1,5,Cooperativa de Quesos 'Las Cabras',Antonio del Valle Saavedra,Export Administrator,Calle del Rosal 4,Oviedo,Asturias,33007,Spain,(98) 598 76 54
2,6,Mayumi's,Mayumi Ohno,Marketing Representative,92 Setsuko Chuo-ku,Osaka,N/A,545,Japan,(06) 431-7877
3,7,Pavlova Ltd.,Ian Devling,Marketing Manager,74 Rose St. Moonie Ponds,Melbourne,Victoria,3058,Australia,(03) 444-2343
4,8,Specialty Biscuits Ltd.,Peter Wilson,Sales Representative,29 King's Way,Manchester,N/A,M14 GSD,UK,(161) 555-4448


### PRODUCTS

In [39]:
#Drop the redundant nested columns
products = products.drop(['supplier', 'category'], axis=1)


In [40]:
#Check for missing values
print(products.isnull().sum())



id                 0
supplierId         0
categoryId         0
quantityPerUnit    0
unitPrice          0
unitsInStock       0
unitsOnOrder       0
reorderLevel       0
discontinued       0
name               0
dtype: int64


In [41]:
#Remane the columns to avoid snake_case
df_products = products.rename(columns={
    "id": "product_id",
    "supplierId": "supplier_id",
    "categoryId": "category_id",
    "quantityPerUnit": "quantity_per_unit",
    "unitPrice": "unit_price",
    "unitsInStock": "units_in_stock",
    "unitsOnOrder": "units_on_order",
    "reorderLevel": "reorder_level",
    "discontinued": "is_discontinued",
    "name": "product_name"
})


In [42]:
df_products.head()

,product_id,supplier_id,category_id,quantity_per_unit,unit_price,units_in_stock,units_on_order,reorder_level,is_discontinued,product_name
0,4,2,3,48 - 6 oz jars,22.00,53,0,0,True,Chef Anton's Cajun Seasoning
1,5,2,2,36 boxes,21.35,0,0,0,True,Chef Anton's Gumbo Mix
2,6,3,2,12 - 8 oz jars,25.00,120,0,25,False,Grandma's Boysenberry Spread
3,7,3,7,12 - 1 lb pkgs.,30.00,15,0,10,False,Uncle Bob's Organic Dried Pears
4,8,3,2,12 - 12 oz jars,40.00,6,0,0,False,Northwoods Cranberry Sauce


## Task 03 Data Loading

### Data Tables Include:
- df_products
- df_orders
- df_suppliers
- df_shippers
- df_customers

In [ ]:
# Load Aiven connection settings from a local .env file (never commit the file).
load_dotenv()

required = ['AIVEN_PASSWORD', 'AIVEN_HOST']
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f'Missing required environment variables: {missing}')

connection_url = URL.create(
    drivername='postgresql+psycopg2',
    username=os.getenv('AIVEN_USER', 'avnadmin'),
    password=os.environ['AIVEN_PASSWORD'],
    host=os.environ['AIVEN_HOST'],
    port=int(os.getenv('AIVEN_PORT', '17508')),
    database=os.getenv('AIVEN_DATABASE', 'sales_dw'),
    query={'sslmode': 'require'},
)
engine = create_engine(connection_url, pool_pre_ping=True)


In [45]:
# 1. push products to db
df_products.to_sql('products', engine, schema='sales', if_exists='replace')


77

In [46]:
# 2. load orders
df_orders.to_sql('orders', engine, schema='sales', if_exists='replace')


830

In [47]:
# 3.load suppliers
df_suppliers.to_sql('suppliers', engine, schema='sales', if_exists='replace')


30

In [48]:
# 4. load customers
df_customers.to_sql('customers', engine, schema='sales', if_exists='replace')


91

In [49]:
# 5. Load shippers
df_shippers.to_sql('shippers', engine, schema='sales', if_exists='replace')

3